In [17]:
import pandas as pd

mmlu_qwen_train_df_easy = pd.read_csv(
    "../../../data/data_splits/entropy_fallback/qwen/train_df_easy.tsv",
    sep="\t",
    header=0,
)
mmlu_qwen_train_df_mid = pd.read_csv(
    "../../../data/data_splits/entropy_fallback/qwen/train_df_middle.tsv",
    sep="\t",
    header=0,
)
mmlu_qwen_train_df_hard = pd.read_csv(
    "../../../data/data_splits/entropy_fallback/qwen/train_df_hard.tsv",
    sep="\t",
    header=0,
)

mmlu_phi4_train_df_easy = pd.read_csv(
    "../../../data/data_splits/entropy_fallback/phi/train_df_easy.tsv",
    sep="\t",
    header=0,
)
mmlu_phi4_train_df_mid = pd.read_csv(
    "../../../data/data_splits/entropy_fallback/phi/train_df_middle.tsv",
    sep="\t",
    header=0,
)
mmlu_phi4_train_df_hard = pd.read_csv(
    "../../../data/data_splits/entropy_fallback/phi/train_df_hard.tsv",
    sep="\t",
    header=0,
)

In [18]:
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer


def count_response_tokens(df, tokenizer: AutoTokenizer, response_column: str):
    token_cnt = 0

    for index, row in tqdm(df.iterrows(), total=df.shape[0]):
        response = row[response_column]

        if type(response) is not str:
            continue

        tokens = tokenizer.encode(response)
        token_cnt += len(tokens)

    return token_cnt


def count_response_tokens_by_split(easy_df, mid_df, hard_df, tokenizer):
    distill_response_column = "distill_response"

    distill_response_token_cnt_easy = count_response_tokens(easy_df, tokenizer, distill_response_column)
    distill_response_token_cnt_mid = count_response_tokens(mid_df, tokenizer, distill_response_column)
    distill_response_token_cnt_hard = count_response_tokens(hard_df, tokenizer, distill_response_column)

    sft_response_column = "answer_index"

    easy_df[sft_response_column] = (easy_df[sft_response_column] + 1).astype(str)
    mid_df[sft_response_column] = (mid_df[sft_response_column] + 1).astype(str)
    hard_df[sft_response_column] = (hard_df[sft_response_column] + 1).astype(str)

    sft_response_token_cnt_easy = count_response_tokens(easy_df, tokenizer, sft_response_column)
    sft_response_token_cnt_mid = count_response_tokens(mid_df, tokenizer, sft_response_column)
    sft_response_token_cnt_hard = count_response_tokens(hard_df, tokenizer, sft_response_column)

    total_distill_token_count = (
        distill_response_token_cnt_easy + distill_response_token_cnt_mid + distill_response_token_cnt_hard
    )
    total_sft_token_count = sft_response_token_cnt_easy + sft_response_token_cnt_mid + sft_response_token_cnt_hard
    pipeline_pt1_token_count = sft_response_token_cnt_easy + sft_response_token_cnt_mid
    pipeline_pt2_token_count = distill_response_token_cnt_hard
    alternative_pt1_token_count = sft_response_token_cnt_mid
    alternative_pt2_token_count = distill_response_token_cnt_easy + distill_response_token_cnt_mid

    return (
        total_sft_token_count,
        total_distill_token_count,
        alternative_pt1_token_count,
        alternative_pt2_token_count,
        pipeline_pt1_token_count,
        pipeline_pt2_token_count,
    )


In [19]:
from transformers import AutoTokenizer

qwen_token_count = count_response_tokens_by_split(
    mmlu_qwen_train_df_easy,
    mmlu_qwen_train_df_mid,
    mmlu_qwen_train_df_hard,
    AutoTokenizer.from_pretrained("Qwen/Qwen2.5-3B-Instruct"),
)

phi4_token_count = count_response_tokens_by_split(
    mmlu_phi4_train_df_easy,
    mmlu_phi4_train_df_mid,
    mmlu_phi4_train_df_hard,
    AutoTokenizer.from_pretrained("microsoft/Phi-4-mini-instruct"),
)


def compare10epochs(
    total_sft_token_count,
    total_distill_token_count,
    alternative_pt1_token_count,
    alternative_pt2_token_count,
    pipeline_pt1_token_count,
    pipeline_pt2_token_count,
):
    sft = total_sft_token_count * 10
    distill = total_distill_token_count * 10
    alternative = alternative_pt1_token_count * 5 + alternative_pt2_token_count * 5
    pipeline = pipeline_pt1_token_count * 5 + pipeline_pt2_token_count * 5
    print(
        f"sft = {sft}, distill = {distill}, alternative = {alternative}, pipeline = {pipeline}, advantage = {1 - pipeline / distill}"
    )


def compare20epochs(
    total_sft_token_count,
    total_distill_token_count,
    alternative_pt1_token_count,
    alternative_pt2_token_count,
    pipeline_pt1_token_count,
    pipeline_pt2_token_count,
):
    sft = total_sft_token_count * 20
    distill = total_distill_token_count * 20
    pipeline = pipeline_pt1_token_count * 10 + pipeline_pt2_token_count * 10
    print(f"sft = {sft}, distill = {distill} pipeline = {pipeline}, advantage = {1 - pipeline / distill}")


print("Qwen 3B:\n")
compare10epochs(*qwen_token_count)
print("Phi4-mini:\n")
compare10epochs(*phi4_token_count)

print("Qwen 3B:\n")
compare20epochs(*qwen_token_count)
print("Phi4-mini:\n")
compare20epochs(*phi4_token_count)


100%|██████████| 900/900 [00:00<00:00, 37219.11it/s]


Qwen 3B:

sft = 29480, distill = 19727980, alternative = 5884085, pipeline = 3994640, advantage = 0.7975139877473517
Phi4-mini:

sft = 27000, distill = 15152310, alternative = 4910895, pipeline = 2678760, advantage = 0.8232111143449414
Qwen 3B:

sft = 58960, distill = 39455960 pipeline = 7989280, advantage = 0.7975139877473517
Phi4-mini:

sft = 54000, distill = 30304620 pipeline = 5357520, advantage = 0.8232111143449414
